In [ ]:
import requests
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Extract from  cancerppd2

In [9]:
import requests, pandas as pd

BASE = "https://webs.iiitd.edu.in/raghava/cancerppd2/api/api.php"

def fetch(params):
    r = requests.get(BASE, params=params, timeout=30)
    r.raise_for_status()
    j = r.json()
    return pd.DataFrame(j.get("data", []))


df_nat = fetch({"dataType": "seq", "dataValue": "Natural"})
df_mod = fetch({"dataType": "seq", "dataValue": "Modified"})
df_cddp2 = pd.concat([df_nat, df_mod], ignore_index=True)


In [10]:
df_cddp2

,id,pmid,year,seq,name,length,lin_cyc,chiral,chem_mod,cter,nter,cell_line,cancer_type,assay,test_time,tissue
0,7722,None,2020,LKKWWKKVKGLLGGLLGKVKKVIK,Seq ID No. 17 from patent ID US202000079827A1,24,Linear,L,None,Free,Free,NCI-H460,Lung Cancer,Tetrazolium-based assay,2-h,Lung
1,7721,None,2020,LKKWWKKVKGLLGGLLGKVKKVIK,Seq ID No. 17 from patent ID US202000079827A1,24,Linear,L,None,Free,Free,HOP-062,Lung Cancer,Tetrazolium-based assay,2-h,Lung
2,7720,None,2020,LKKWWKKVKGLLGGLLGKVKSVIK,Seq ID No. 16 from patent ID US202000079827A1,24,Linear,L,None,Free,Free,Calu-1,Lung Cancer,Tetrazolium-based assay,2-h,Lung
3,7719,None,2020,LKKWWKKVKGLLGGLLGKVKSVIK,Seq ID No. 16 from patent ID US202000079827A1,24,Linear,L,None,Free,Free,NCI-H460,Lung Cancer,Tetrazolium-based assay,2-h,Lung
4,7718,None,2020,LKKWWKKVKGLLGGLLGKVKSVIK,Seq ID No. 16 from patent ID US202000079827A1,24,Linear,L,None,Free,Free,HOP-062,Lung Cancer,Tetrazolium-based assay,2-h,Lung
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6113,4676,None,2016,LTFAEYWAQLaAAAAAa,ALRN-6924,17,Linear,Mix,staple between Ala4 and d-Ala11,Free,Free,TP53-WT subcutaneous mouse xenograft models,Breast Cancer,Not Available,28 days,Breast
6114,7851,29580912,2018,ywNTF(AzGly)R(Me)W,TAK-448,8,Linear,Mix,aza-glycine; Me=Methylation,Amidation,Acetylation,VCaP,Prostate Cancer,Given i.h.; dosings on day 0 and 28,28 days,Prostate
6115,7848,None,2018,ACSAp(Dab)RYCYQKpPYH,Balixafortide,16,Cyclic ; Disulfide bridge: Cys2-Cys9,Mix,"Dab = 2,4-diaminobutyric acid",Free,Free,Namalwa,Lymphoma,HUVEC sprouting,Not available,Blood
6116,7849,None,2018,ACSAp(Dab)RYCYQKpPYH,Balixafortide,16,Cyclic ; Disulfide bridge: Cys2-Cys9,Mix,"Dab = 2,4-diaminobutyric acid",Free,Free,Jurkat,Blood Cancer,HUVEC sprouting,Not available,Blood


In [11]:
df_cddp2.to_csv('/content/drive/MyDrive/TGPepGM/raw_datasets/CancerPPD2.csv', index=False, encoding="utf-8")


# Extract from dracp

In [12]:
# ----- dracp_figshare_download.py -----
import os, io, requests, pandas as pd

ARTICLE_ID = 23633877   # DRACP article id on Figshare (from the URL)
API = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}"
OUT_DIR = "/content/drive/MyDrive/TGPepGM/raw_datasets/dracp"
os.makedirs(OUT_DIR, exist_ok=True)

j = requests.get(API, timeout=30).json()
files = j.get("files", [])  # each has name, size, md5, download_url

tables = []
for f in files:
    url, name = f["download_url"], f["name"]
    path = os.path.join(OUT_DIR, name)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, "wb") as w:
            for chunk in r.iter_content(1<<20):
                if chunk: w.write(chunk)
    # try to load into pandas for convenience
    try:
        if name.lower().endswith(".csv"):
            df = pd.read_csv(path)
        elif name.lower().endswith((".xlsx", ".xls")):
            df = pd.read_excel(path)
        else:
            continue
        df["__source_file"] = name
        tables.append(df)
    except Exception:
        pass

if tables:
    big = pd.concat(tables, ignore_index=True)
    big.to_csv(os.path.join(OUT_DIR, "dracp_merged.csv"), index=False)
    print(f"Merged table shape: {big.shape}")
else:
    print("Downloaded files; none were recognized as CSV/Excel.")


Merged table shape: (14500, 86)


In [13]:
dracp_copy = big.copy()

In [14]:
dracp_copy.dropna(subset=['Sequence'], inplace=True)
#dracp_copy.drop_duplicates(subset=["DCTPep_ID"]).reset_index(drop=True)


In [15]:
dracp_copy

,DCTPep_ID,DRAMP_ID,CancerPPD_ID,DBAASP_ID,Cppsite_ID,Peptide_Name,Sequence,Sequence_Length,UniProt_ID,PubChem_CID,...,Dosage_Form/Route,Company,Marketing_Status,Drug_ID,Approval_year,ClinicalTrials.gov_Identifier,Title,Condition_or_disease,Phase,Purpose
0,DCTPep00001,DRAMP02912,Not available,1485,Not available,SMAP-29,RGLRRLGRKIAHGVKKYGPTVLRIIRIA,28,Not available,16130512,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DCTPep00002,Not available,Not available,Not available,Not available,CA-MA,KWKLFKKIGIGKFLHSAKKF,20,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DCTPep00003,Not available,Not available,Not available,Not available,CA-MA3,KWKLFKKIGPGKFLHSAKKF,20,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DCTPep00004,Not available,Not available,Not available,Not available,CA-MA1,KWKLFKKIKFLHSAKKF,17,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DCTPep00005,Not available,Not available,Not available,Not available,CA-MA2,KWKLFKKIPKFLHSAKKF,18,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6102,DCTPep06890,Not available,Not available,Not available,Not available,AC-CCSP-3-p,GLFAVⓍKKVⓍSVIKGL,16,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6103,DCTPep06891,Not available,Not available,Not available,Not available,AC-CCSP-4-o,GLFAVIKKⓍASVⓍKGL,16,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6104,DCTPep06892,Not available,Not available,Not available,Not available,AC-CCSP-4-m,GLFAVIKKⓍASVⓍKGL,16,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6105,DCTPep06893,Not available,Not available,Not available,Not available,AC-CCSP-4-p,GLFAVIKKⓍASVⓍKGL,16,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
dracp_copy.to_csv("/content/drive/MyDrive/TGPepGM/raw_datasets/dracp.csv", index=False)


# Extract from DCTPep

we downloaded the dataset from [DCTPep](http://dctpep.cpu-bioinfor.org/downloads/)